In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models

def inception_module(x, filters):
    # 1x1 Convolution
    conv1x1 = layers.Conv2D(filters[0], (1, 1), padding='same', activation='relu')(x)
    
    # 3x3 Convolution
    conv3x3 = layers.Conv2D(filters[1], (1, 1), padding='same', activation='relu')(x)
    conv3x3 = layers.Conv2D(filters[2], (3, 3), padding='same', activation='relu')(conv3x3)
    
    # 5x5 Convolution
    conv5x5 = layers.Conv2D(filters[3], (1, 1), padding='same', activation='relu')(x)
    conv5x5 = layers.Conv2D(filters[4], (5, 5), padding='same', activation='relu')(conv5x5)
    
    # MaxPooling
    maxpool = layers.MaxPooling2D((3, 3), strides=(1, 1), padding='same')(x)
    maxpool = layers.Conv2D(filters[5], (1, 1), padding='same', activation='relu')(maxpool)
    
    # Concatenate
    return layers.Concatenate()([conv1x1, conv3x3, conv5x5, maxpool])

def googlenet(input_shape, num_classes):
    input_layer = layers.Input(shape=input_shape)
    
    x = layers.Conv2D(64, (7, 7), strides=(2, 2), padding='same', activation='relu')(input_layer)
    x = layers.MaxPooling2D((3, 3), strides=(2, 2), padding='same')(x)
    
    x = layers.Conv2D(64, (1, 1), padding='same', activation='relu')(x)
    x = layers.Conv2D(192, (3, 3), padding='same', activation='relu')(x)
    x = layers.MaxPooling2D((3, 3), strides=(2, 2), padding='same')(x)
    
    # Inception Modules
    x = inception_module(x, [64, 96, 128, 16, 32, 32])  # Inception 3a
    x = inception_module(x, [128, 128, 192, 32, 96, 64])  # Inception 3b
    x = layers.MaxPooling2D((3, 3), strides=(2, 2), padding='same')(x)
    
    x = inception_module(x, [192, 96, 208, 16, 48, 64])  # Inception 4a
    x = inception_module(x, [160, 112, 224, 24, 64, 64])  # Inception 4b
    x = inception_module(x, [128, 128, 256, 24, 64, 64])  # Inception 4c
    x = inception_module(x, [112, 144, 288, 32, 64, 64])  # Inception 4d
    x = inception_module(x, [256, 160, 320, 32, 128, 128])  # Inception 4e
    x = layers.MaxPooling2D((3, 3), strides=(2, 2), padding='same')(x)
    
    x = inception_module(x, [256, 160, 320, 32, 128, 128])  # Inception 5a
    x = inception_module(x, [384, 192, 384, 48, 128, 128])  # Inception 5b
    
    # Global Average Pooling and Fully Connected Layer
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    output_layer = layers.Dense(num_classes, activation='softmax')(x)
    model = models.Model(inputs=input_layer, outputs=output_layer)
    return model

input_shape = (224, 224, 3)  
num_classes = 1000  
model = googlenet(input_shape, num_classes)
model.summary()

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_2 (InputLayer)           [(None, 224, 224, 3  0           []                               
                                )]                                                                
                                                                                                  
 conv2d_57 (Conv2D)             (None, 112, 112, 64  9472        ['input_2[0][0]']                
                                )                                                                 
                                                                                                  
 max_pooling2d_13 (MaxPooling2D  (None, 56, 56, 64)  0           ['conv2d_57[0][0]']              
 )                                                                                          